### DHA Lahore Real Estate Market Analysis

This notebook analyzes real estate data from DHA Lahore to understand property prices, trends, and market segments. The analysis includes web scraping, data cleaning, statistical analysis, and machine learning modeling.


### Data Collection Setup

We'll be scraping property listings from Zameen.com, focusing on DHA Lahore area.

In [1]:
# Importing libraries
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
import statistics


In [2]:
# Setting up Selenium with headless Chrome
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

In [3]:
# Setup browser (headless Chrome)
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(), options=chrome_options)

records = []

for pg in range(1, 6):
    url = f"https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-{pg}.html"
    print(f"[+] Processing page {pg}: {url}")
    driver.get(url)

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "li[aria-label='Listing']"))
    )

    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.find_all("li", {"aria-label": "Listing"})
    print(f"  ↳ Listings found: {len(items)}")

    if not items:
        print(f"  [!] No listings detected on page {pg}.")
        continue

    for item in items:
        try:
            loc = item.find("div", class_=lambda x: x and "location" in x.lower())
            loc = loc.text.strip() if loc else "N/A"

            price_tag = item.find("span", {"aria-label": "Price"})
            price = price_tag.text.strip() if price_tag else "N/A"

            beds_tag = item.find("span", {"aria-label": "Beds"})
            beds = beds_tag.text.strip() if beds_tag else "N/A"

            baths_tag = item.find("span", {"aria-label": "Baths"})
            baths = baths_tag.text.strip() if baths_tag else "N/A"

            area_tag = item.find("span", {"aria-label": "Area"})
            area = area_tag.text.strip() if area_tag else "N/A"

            records.append(
                {
                    "Location": loc,
                    "Price": price,
                    "Bedrooms": beds,
                    "Bathrooms": baths,
                    "Area": area,
                }
            )
        except Exception as err:
            print(f"  [x] Skipped one item due to error: {err}")

driver.quit()

df = pd.DataFrame(records)
print(f"\nTotal properties scraped: {len(records)}")
df.to_csv("dha_lahore_houses.csv", index=False)
print("File saved: dha_lahore_houses.csv")

df.head()

[+] Processing page 1: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-1.html
  ↳ Listings found: 27
[+] Processing page 2: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-2.html
  ↳ Listings found: 27
[+] Processing page 3: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-3.html
  ↳ Listings found: 27
[+] Processing page 4: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-4.html
  ↳ Listings found: 27
[+] Processing page 5: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-5.html
  ↳ Listings found: 27

Total properties scraped: 135
File saved: dha_lahore_houses.csv


,Location,Price,Bedrooms,Bathrooms,Area
0,N/A,10.5 Crore,5,6,1 Kanal
1,N/A,6.5 Crore,5,6,1 Kanal
2,N/A,7.75 Crore,5,5,1 Kanal
3,N/A,7.15 Crore,5,6,1 Kanal
4,N/A,8.74 Crore,5,6,1 Kanal


In [4]:
# Load dataset
try:
    listings_df = pd.read_csv("dha_lahore_houses.csv")
except FileNotFoundError:
    print("File not found: 'dha_lahore_houses.csv'. Please make sure it exists.")
    exit()


def convert_price(text):
    if pd.isna(text) or text == "N/A":
        return np.nan

    text = str(text).upper().strip()
    value_match = re.search(r"(\d+(\.\d+)?)", text)
    if not value_match:
        return np.nan

    amount = float(value_match.group(1))

    unit_match = re.search(r"(CRORE|LAC|LAKH|MILLION|THOUSAND)", text)
    if unit_match:
        unit = unit_match.group(1).lower()
        if "crore" in unit:
            amount *= 10_000_000
        elif "lakh" in unit or "lac" in unit:
            amount *= 100_000
        elif "million" in unit:
            amount *= 1_000_000
        elif "thousand" in unit:
            amount *= 1_000

    return amount


def convert_area(text):
    if pd.isna(text) or text == "N/A":
        return np.nan

    text = str(text).strip()
    match = re.search(
        r"(\d+(\.\d+)?)\s*(Marla|Kanal|Sq\.?\s*Ft\.?|Sq\.?\s*Yd\.?|Square\s*Foot|Square\s*Yard)",
        text,
        re.I,
    )
    if not match:
        return np.nan

    size = float(match.group(1))
    unit = match.group(3).lower()

    if "marla" in unit:
        size *= 272.25
    elif "kanal" in unit:
        size *= 5445
    elif "sq yd" in unit or "square yard" in unit:
        size *= 9
    # Square feet assumed as-is

    return size


def detect_phase(location):
    if pd.isna(location) or location == "N/A":
        return "Unknown"

    location = str(location).upper()

    match_phase = re.search(r"PHASE\s+(\d+)", location)
    if match_phase:
        return f"Phase {match_phase.group(1)}"

    match_town = re.search(r"DHA\s+(\d+)\s*TOWN", location)
    if match_town:
        return f"Phase {match_town.group(1)} Town"

    return "Unknown"


# --- Apply transformations ---

listings_df["Price_PKR"] = listings_df["Price"].apply(convert_price)
listings_df["Area_SqFt"] = listings_df["Area"].apply(convert_area)
listings_df["Phase"] = listings_df["Location"].apply(detect_phase)

# Drop incomplete rows
filtered_df = listings_df.dropna(subset=["Price_PKR", "Area_SqFt"]).copy()

# Price per sq ft calculation
filtered_df["Rate_Per_SqFt"] = filtered_df["Price_PKR"] / filtered_df["Area_SqFt"]

# Aggregate average rates by phase
phase_summary = (
    filtered_df.groupby("Phase")["Rate_Per_SqFt"].agg(["mean", "count", "std"]).round(2)
)
phase_summary = phase_summary[phase_summary["count"] >= 3]

# --- Output ---

print("\nAverage Price per Square Foot (by Phase):")
print(phase_summary)

if not phase_summary.empty:
    top_value_area = phase_summary["mean"].idxmin()
    lowest_avg = phase_summary.loc[top_value_area, "mean"]
    print(
        f"\n Most affordable phase: {top_value_area} → Avg PKR {lowest_avg:,.2f} per sq ft"
    )
else:
    print(
        "\nNot enough data to compute reliable averages (minimum 3 listings per phase)."
    )


Average Price per Square Foot (by Phase):
            mean  count      std
Phase                           
Unknown  17989.8    135  7418.23

 Most affordable phase: Unknown → Avg PKR 17,989.80 per sq ft


### Data Preprocessing and Initial Analysis

The analysis will help identify the most affordable areas within DHA Lahore.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# --- Load Data ---
try:
    houses_df = pd.read_csv("dha_lahore_houses.csv")
except FileNotFoundError:
    print(
        "File 'dha_lahore_houses.csv' not found. Make sure it's in the working directory."
    )
    exit()

# --- Data Processing Functions ---


def clean_price(text):
    if pd.isna(text) or text == "N/A":
        return np.nan
    text = str(text).upper().strip()
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if not match:
        return np.nan
    value = float(match.group(1))
    unit = re.search(r"(CRORE|LAC|LAKH|MILLION|THOUSAND)", text)
    if unit:
        unit = unit.group(1).lower()
        if "crore" in unit:
            value *= 10_000_000
        elif "lakh" in unit or "lac" in unit:
            value *= 100_000
        elif "million" in unit:
            value *= 1_000_000
        elif "thousand" in unit:
            value *= 1_000
    return value


def clean_area(area_text):
    if pd.isna(area_text) or area_text == "N/A":
        return np.nan
    area_text = str(area_text).strip()
    match = re.search(r"(\d+(?:\.\d+)?)\s*(Marla|Kanal|Sq\.?\s*Ft\.?)", area_text, re.I)
    if not match:
        return np.nan
    value = float(match.group(1))
    unit = match.group(2).lower()
    if "marla" in unit:
        value *= 272.25
    elif "kanal" in unit:
        value *= 5445
    return value


def extract_numeric(val):
    if pd.isna(val) or val == "N/A":
        return np.nan
    try:
        return int(re.search(r"\d+", str(val)).group())
    except:
        return np.nan


def get_phase(location):
    if pd.isna(location) or location == "N/A":
        return "Unknown"
    location = str(location).upper()
    match = re.search(r"PHASE\s+(\d+)", location)
    if match:
        return f"Phase {match.group(1)}"
    alt_match = re.search(r"DHA\s+(\d+)\s*TOWN", location)
    if alt_match:
        return f"Phase {alt_match.group(1)} Town"
    return "Unknown"


# --- Apply Cleaning ---
houses_df["Price_PKR"] = houses_df["Price"].apply(clean_price)
houses_df["Area_SqFt"] = houses_df["Area"].apply(clean_area)
houses_df["Bedrooms"] = houses_df["Bedrooms"].apply(extract_numeric)
houses_df["Bathrooms"] = houses_df["Bathrooms"].apply(extract_numeric)
houses_df["Phase"] = houses_df["Location"].apply(get_phase)

# --- Filter Out Incomplete Rows ---
clean_df = houses_df.dropna(
    subset=["Price_PKR", "Area_SqFt", "Bedrooms", "Bathrooms", "Phase"]
)

# --- Feature Selection ---
features = clean_df[["Bedrooms", "Bathrooms", "Area_SqFt", "Phase"]]
target = clean_df["Price_PKR"]

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

# --- Preprocessing + Model Pipeline ---
preprocess = ColumnTransformer(
    transformers=[("phase_ohe", OneHotEncoder(handle_unknown="ignore"), ["Phase"])],
    remainder="passthrough",
)

model_pipeline = Pipeline([("prep", preprocess), ("reg", LinearRegression())])

# --- Train Model ---
model_pipeline.fit(X_train, y_train)

# --- Make Predictions & Evaluate ---
y_pred = model_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nLinear Regression Performance:")
print(f" - Mean Squared Error : {mse:,.0f}")
print(f" - R² Score           : {r2:.4f}")

# --- Coefficient Interpretation ---
feature_labels = model_pipeline.named_steps["prep"].get_feature_names_out()
coeffs = model_pipeline.named_steps["reg"].coef_

coef_df = pd.DataFrame({"Feature": feature_labels, "Impact": coeffs}).sort_values(
    by="Impact", ascending=False
)

print("\nFeature Influence on Price (High → Low):")
print(coef_df.to_string(index=False))


Linear Regression Performance:
 - Mean Squared Error : 989,660,586,896,763
 - R² Score           : -0.3818

Feature Influence on Price (High → Low):
                 Feature        Impact
     remainder__Bedrooms  2.562333e+07
    remainder__Area_SqFt  2.462167e+04
phase_ohe__Phase_Unknown  0.000000e+00
    remainder__Bathrooms -2.649138e+07


### Predictive Modeling with Linear Regression

The model will help us understand which factors have the strongest influence on property prices in DHA Lahore.

In [6]:
df = pd.read_csv("dha_lahore_houses.csv")


# Function to parse price to PKR (numerical)
def parse_price(price_str):
    if pd.isna(price_str) or price_str == "N/A":
        return np.nan
    price_str = str(price_str).upper().strip()
    num_match = re.search(r"(\d+(?:\.\d+)?)", price_str)
    if not num_match:
        return np.nan
    num = float(num_match.group(1))
    unit = re.search(r"(CRORE|LAC|LAKH|MILLION|THOUSAND)", price_str)
    if unit:
        unit = unit.group(1).lower()
        if "crore" in unit:
            num *= 10000000
        elif "lac" in unit or "lakh" in unit:
            num *= 100000
        elif "million" in unit:
            num *= 1000000
        elif "thousand" in unit:
            num *= 1000
    return num


# Apply parsing
df["Price_PKR"] = df["Price"].apply(parse_price)

# Drop rows with NaN prices
df_clean = df.dropna(subset=["Price_PKR"])


# Function to categorize based on price
def categorize_price(price):
    if price < 30000000:  # < 3 Crore
        return "Affordable"
    elif 30000000 <= price <= 60000000:  # 3-6 Crore
        return "Mid-Range"
    else:  # > 6 Crore
        return "Luxury"


# Apply categorization
df_clean["Category"] = df_clean["Price_PKR"].apply(categorize_price)

# Calculate percentage distribution
distribution = df_clean["Category"].value_counts(normalize=True) * 100
distribution = distribution.round(2).reset_index()
distribution.columns = ["Category", "Percentage"]

# Display the distribution
print("Percentage Distribution of Properties by Category:")
print(distribution.to_string(index=False))

Percentage Distribution of Properties by Category:
  Category  Percentage
    Luxury       64.44
Affordable       20.74
 Mid-Range       14.81


### Market Segmentation Analysis

This final section segments the property market into three categories:
- Affordable: Properties under 3 Crore PKR
- Mid-Range: Properties between 3-6 Crore PKR
- Luxury: Properties above 6 Crore PKR

This segmentation helps understand the distribution of properties across different price ranges and identify market opportunities.